## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 \
pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 147.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 301.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 340.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 294.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.28-cp312-cp312-linux_x86_64.whl size=8960012 sha256=f5dc615002485eab0b0fecda14fa7fadc9deb360622585a904a5064d1300b624
  Stored in directory: /tmp/pip-ephem-wheel-cache-6tnlvu2y/wheels/f6/3d/56/3f3ff71bc61e1aa756c6d6fcd36905032bc5868d3a5bd39d60
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Succes

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.3 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [4]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [5]:
model_path = hf_hub_download(
  repo_id=model_name_or_path,
  filename=model_basename
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
llm = Llama(
  model_path=model_path,
  n_ctx=2048,
  n_gpu_layers=-1,
  n_batch=256,
  verbose=True
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [7]:
def response(query, max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    prompt = f"<s>[INST] {query} [/INST]"

    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    return output["choices"][0]["text"].strip()

In [8]:
print(response("What treatment options are available for managing low blood sugar?"));

Low blood sugar, also known as hypoglycemia, can be caused by various conditions including diabetes, certain medications, or other health issues. Here are some common treatment options for managing low blood sugar:

1. Consuming carbohydrates: The quickest way to raise your blood sugar level is by consuming foods that contain carbohydrates. These include fruits, juices, sweets, and starches like bread, rice, or pasta. Aim for 15-20 grams of carbohydrates, which is roughly equivalent to a medium apple, a cup of milk, or 3-4 glucose tablets.

2. Glucose tablets or gel: These are readily available over the counter and can be carried with you for quick treatment of low blood sugar. They provide a fast source of glucose to raise your blood sugar level.

3. Dextrose solution: Injectable dextrose solutions, such as D50W (50% dextrose in water), are prescribed by doctors for severe cases of hypoglycemia or when oral treatments are not effective. These should only be used under medical supervis

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [9]:
user_query_1 = "What is the protocol for managing sepsis in a critical care unit?";
user_response_1  = response(user_query_1)
print(user_response_1)

Llama.generate: prefix-match hit


Sepsis is a life-threatening condition that requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis early, which may include fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. Quick initiation of antibiotics: Start appropriate broad-spectrum antibiotics as soon as possible based on the patient's clinical presentation and culture results if available.
3. Fluid resuscitation: Aggressively administer intravenous fluids to maintain adequate tissue perfusion, with a goal of maintaining a mean arterial pressure (MAP) of 65 mmHg or higher and a central venous oxygen saturation (ScvO2) of greater than 70%.
4. Vasopressor support: If the patient is hypotensive despite adequate fluid resuscitation, administer vasopressors to maintain MAP.
5. Corticosteroids: Conside

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [10]:
user_query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?";
user_response_2  = response(user_query_2)
print(user_response_2)

Llama.generate: prefix-match hit


Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms of appendicitis:

1. Abdominal pain: The pain typically starts as a mild discomfort around the navel or upper abdomen that may come and go. However, it can quickly develop into a sharp, persistent pain in the lower right side of the abdomen.
2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and discomfort.
3. Nausea and vomiting: Vomiting is common in appendicitis, especially as the condition progresses.
4. Fever: A fever of 100.4°F (38°C) or higher is a common symptom of appendicitis.
5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. Inability to pass gas: Passing gas can be difficult due to the pressure on the i

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [11]:
user_query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?";
user_response_3  = response(user_query_3)
print(user_response_3)

Llama.generate: prefix-match hit


Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other areas of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Here are some effective treatments and possible causes for this condition:

1. Corticosteroids: Corticosteroids are the most common treatment for alopecia areata. They work by reducing inflammation and suppressing the immune system, which helps to promote hair regrowth. Corticosteroids can be applied topically or taken orally in the form of pills or injections.
2. Immunotherapy: Immunotherapies such as diphenylcyclopropenone (DPCP) and squaric acid dibutylester (SADBE) are used to stimulate an immune response that promotes hair regrowth. These treatments can be applied topically or given as injections.
3. Minoxidil: Minoxidil is a medication that is commonly used to treat male and female pattern baldness, but 

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [12]:
user_query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?";
user_response_4  = response(user_query_4)
print(user_response_4)

Llama.generate: prefix-match hit


Treatment for a physical injury to the brain can depend on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: For severe brain injuries, immediate medical attention is necessary. This may include surgery to remove hematomas or other obstructions, as well as measures to prevent further damage, such as controlling blood pressure and maintaining adequate oxygen levels.
2. Rehabilitation: Rehabilitation is an essential part of treatment for brain injuries. It can help improve function, reduce symptoms, and promote independence. Rehabilitation may include physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation.
3. Medications: Depending on the specific symptoms of the brain injury, medications may be prescribed to manage conditions such as seizures, pain, or depression.
4. Assistive devices: Devices such as wheelchairs, walkers, or communication aids can help individuals with brain injuries regai

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [13]:
user_query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?";
user_response_5  = response(user_query_5)
print(user_response_5)

Llama.generate: prefix-match hit


If a person has fractured their leg while hiking, it is important to follow these necessary precautions and treatment steps:

1. Assess the situation: Check the severity of the injury and assess if the fracture is open or closed. An open fracture is when the bone breaks through the skin, which increases the risk of infection. If the fracture is open, do not attempt to move the person as it may cause further damage or infection.
2. Call for help: If possible, call for emergency medical assistance. If there is no cell phone reception, use a satellite phone or hike back to civilization to seek help.
3. Provide first aid: Keep the person calm and comfortable. Apply a sterile dressing to any open wounds and apply ice to reduce swelling. Do not try to realign the bone as it may cause further damage. Splint the leg if possible using a makeshift splint or a commercial one if available.
4. Transportation: If the person is stable, transport them to the nearest medical facility as soon as possibl

## Question Answering using LLM with Prompt Engineering

In [14]:
prompt_template = """
You are a medical AI assistant. Provide accurate, evidence-based, and clinically sound answers grounded in established medical guidelines and best practices.

Guidelines:
- Prioritize factual accuracy and medical consensus.
- Use clear, concise language appropriate for both healthcare professionals and general audiences.
- Structure responses logically (e.g., definition, causes, symptoms, diagnosis, treatment when relevant).
- Avoid unnecessary verbosity or speculation.
- If information is uncertain, incomplete, or outside your knowledge scope, explicitly acknowledge the limitation.

Safety:
- Do not provide diagnoses or treatment plans as a substitute for professional medical care.
- Encourage seeking qualified healthcare professionals when appropriate.

Answer the question directly and clearly.
"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [15]:
query_1 = prompt_template + "\n" + "What is the protocol for managing sepsis in a critical care unit?";
print(response(query_1))

Llama.generate: prefix-match hit


Sepsis is a life-threatening condition caused by the body's response to an infection, which can lead to tissue damage, organ failure, and even death if not recognized and treated promptly. In a critical care unit, managing sepsis involves a systematic approach based on established guidelines from organizations like the Surviving Sepsis Campaign (SSC). Here are the key steps:

1. **Recognition**: Suspect sepsis in any patient with suspected or confirmed infection and signs of organ dysfunction such as altered mental status, respiratory distress, lactic acidosis, or low urine output.

2. **Early goal-directed therapy (EGDT)**: Initiate resuscitation measures within 1 hour of recognition:
   - Administer oxygen to maintain adequate oxygenation and ventilate if necessary.
   - Provide intravenous fluids to restore intravascular volume, aiming for a mean arterial pressure (MAP) >65 mmHg and central venous pressure (CVP) 8-12 cm H2O or a MAP >70 mmHg and CVP <12 cm H2O if no CVP measurement 

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [16]:
query_2 = prompt_template + "\n" + "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?";
print(response(query_2))

Llama.generate: prefix-match hit


Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right abdomen. Common symptoms include:

1. Abdominal pain: The pain typically starts around the navel area and then shifts to the lower right side of the abdomen. It may be constant or intermittent and worsens with movement, coughing, or sneezing.
2. Loss of appetite: Affected individuals may experience a decrease in appetite due to abdominal discomfort.
3. Nausea and vomiting: These symptoms are common in appendicitis and can occur due to the irritation of the appendix or as a response to the body's inflammatory response.
4. Fever: A fever may be present, with temperatures ranging from 100°F (37.8°C) to 103°F (39.4°C).
5. Rebound tenderness: This is a sign of peritonitis, an inflammation of the peritoneum, which may occur when appendicitis ruptures. When pressure is applied to the abdomen during a physical examination, there is pain and discomfort in respo

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [17]:
query_3 = prompt_template + "\n" + "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?";
print(response(query_3))

Llama.generate: prefix-match hit


Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that results in hair loss from specific areas of the scalp or other parts of the body. The exact cause of alopecia areata is not fully understood, but it's believed to be related to a problem with the immune system.

Possible causes of sudden patchy hair loss include:
1. Genetics: There seems to be a higher risk for developing alopecia areata if there is a family history of the condition.
2. Stress: Emotional or physical stress can trigger the onset of alopecia areata in some people.
3. Viral infections: Some viruses, such as Epstein-Barr virus and human immunodeficiency virus (HIV), have been linked to alopecia areata.
4. Other autoimmune diseases: People with conditions like thyroid disease, vitiligo, or lupus may be more likely to develop alopecia areata.
5. Certain medications: Some drugs, such as certain types of antidepressants and nonsteroidal anti-inflammatory drugs (NSAIDs), have been associated

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [18]:
query_4 = prompt_template + "\n" + "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?";
print(response(query_4))

Llama.generate: prefix-match hit


A brain injury, also known as traumatic brain injury (TBI), can result from various causes such as motor vehicle accidents, falls, sports injuries, or violence. Treatment for a brain injury depends on the severity and specific type of injury. Here are some common treatments:

1. Initial care: For acute brain injuries, the primary focus is on ensuring the person's safety, maintaining adequate breathing and circulation, and preventing further harm. This may involve emergency surgery to remove hematomas or decompress skull fractures, intubation for airway protection, and administration of medications to manage increased intracranial pressure or seizures.

2. Rehabilitation: Rehabilitation is a crucial component of brain injury treatment. It can help improve functional abilities, reduce symptoms, and promote recovery. Rehabilitation may include physical therapy, occupational therapy, speech-language therapy, cognitive rehabilitation, and psychosocial support.

3. Medications: Depending on 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [19]:
query_5 = prompt_template + "\n" + "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?";
print(response(query_5))

Llama.generate: prefix-match hit


A leg fracture is a broken bone in the leg, which can occur due to various reasons such as falling or twisting the leg during an activity like hiking. Here are some necessary precautions and treatment steps for a person who has fractured their leg:

1. Assess the severity of the injury: If you suspect a leg fracture, it's essential to assess the extent of the injury. Check for signs of swelling, bruising, deformity, or inability to move the leg. If there is significant pain, instability, or open wounds, seek medical attention immediately.
2. Immobilize the affected limb: Use a splint, sling, or brace to immobilize the fractured leg to prevent further damage and promote healing. Be sure not to force the bone back into place as this could cause more harm.
3. Control bleeding: Apply direct pressure with a clean cloth to control any bleeding if present. Elevate the injured leg above heart level to reduce swelling and pain.
4. Pain management: Over-the-counter pain relievers like acetaminop

## Data Preparation for RAG

In [20]:
#Libraries for processing dataframes, text
import json, os
import tiktoken
import pandas as pd

#Libraries for Londing Data, Chunking, Embedding, and Vector Databases
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Loading the Data

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
pdf_loader = PyMuPDFLoader("/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf")

In [23]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [24]:
for i in range(5):
    print(f"Page {i+1}:", end='\n')
    print(manual[i].page_content, end='\n')

Page 1:
emmaomoh@gmail.com
R4K1SHWP6J
This file is meant for personal use by emmaomoh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page 2:
emmaomoh@gmail.com
R4K1SHWP6J
This file is meant for personal use by emmaomoh@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page 3:
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...................................................................................................................................................................

#### Checking the number of pages

In [25]:
print(f"Number of pages: {len(manual)}")

Number of pages: 4114


### Data Chunking

In [26]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1024,
    chunk_overlap=64,
)

In [27]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [28]:
len(document_chunks)

4533

In [29]:
document_chunks[0].page_content

'emmaomoh@gmail.com\nR4K1SHWP6J\nThis file is meant for personal use by emmaomoh@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [30]:
document_chunks[1].page_content

'emmaomoh@gmail.com\nR4K1SHWP6J\nThis file is meant for personal use by emmaomoh@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [31]:
document_chunks[2].page_content

"Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [32]:
document_chunks[3].page_content

'491\nChapter 44. Foot & Ankle Disorders    .....................................................................................................................................\n502\nChapter 45. Tumors of Bones & Joints    ...............................................................................................................................\n510\n5 - Ear, Nose, Throat & Dental Disorders    ..................................................................................................................\n510\nChapter 46. Approach to the Patient With Ear Problems    ...........................................................................................\n523\nChapter 47. Hearing Loss    .........................................................................................................................................................\n535\nChapter 48. Inner Ear Disorders    ...................................................................................................

In [33]:
document_chunks[4].page_content

'921\nChapter 94. Adrenal Disorders    ................................................................................................................................................\n936\nChapter 95. Polyglandular Deficiency Syndromes    ........................................................................................................\n939\nChapter 96. Porphyrias    ..............................................................................................................................................................\n949\nChapter 97. Fluid & Electrolyte Metabolism    .....................................................................................................................\n987\nChapter 98. Acid-Base Regulation & Disorders    ..............................................................................................................\n1001\nChapter 99. Diabetes Mellitus & Disorders of Carbohydrate Metabolism    ..................................................

### Embedding

In [34]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_18574/4198310515.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [35]:
embedding_1 = embedding_model.embed_query(document_chunks[1].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[2].page_content)

In [36]:
print("Dimension of embedding vector ", len(embedding_1))
len(embedding_1) == len(embedding_2)

Dimension of embedding vector  1024


True

In [37]:
embedding_1, embedding_2

([-0.024395890533924103,
  -0.013754426501691341,
  -0.02125728502869606,
  -0.0002551517100073397,
  -0.013191153295338154,
  -0.03293708711862564,
  0.001685615163296461,
  0.04607722535729408,
  0.021802658215165138,
  0.00155170529615134,
  0.0192253440618515,
  -0.0006206110119819641,
  0.015062721446156502,
  -0.05363303795456886,
  0.0068460265174508095,
  -0.001016263384371996,
  0.0005286958767101169,
  -0.03837760537862778,
  0.002057036617770791,
  -0.0017022971296682954,
  -0.0017505843425169587,
  0.00046893066610209644,
  -0.0729655772447586,
  -0.03898443654179573,
  0.003066563978791237,
  0.03032122738659382,
  0.023456260561943054,
  -0.030435234308242798,
  0.055492665618658066,
  0.03240329027175903,
  -0.0359351709485054,
  -0.0020562876015901566,
  0.021706828847527504,
  -0.04153513163328171,
  0.0036655424628406763,
  -0.027470864355564117,
  0.06149696186184883,
  -0.010942834429442883,
  -0.04322343319654465,
  -0.050836145877838135,
  -0.011081208474934101,
 

### Vector Database

In [38]:
output_dir = "/content/drive/MyDrive/AIML/vector_store"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [40]:
vector_store = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=output_dir
)

In [41]:
vector_store = Chroma(
    persist_directory=output_dir,
    embedding_function=embedding_model
)

/tmp/ipykernel_18574/1354474335.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


In [42]:
vector_store.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [43]:
vector_store.similarity_search("What is the protocol for managing sepsis in a critical care unit?", k=1)

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'trapped': '', 'format': 'PDF 1.7', 'keywords': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'file_path': '/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf', 'author': '', 'page': 2400, 'subject': '', 'creator': 'Atop CHM to PDF Converter', 'total_pages': 4114, 'modDate': 'D:20260325043229Z', 'creationDate': 'D:20120615054440Z', 'moddate': '2026-03-25T04:32:29+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients

In [44]:
vector_store.similarity_search("What is the protocol for managing sepsis in a critical care unit?", k=3)

[Document(metadata={'modDate': 'D:20260325043229Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'total_pages': 4114, 'file_path': '/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'keywords': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'page': 2400, 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'creator': 'Atop CHM to PDF Converter', 'moddate': '2026-03-25T04:32:29+00:00', 'source': '/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf', 'author': '', 'format': 'PDF 1.7', 'subject': '', 'trapped': ''}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients

### Retriever

In [45]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [46]:
relevant_documents = retriever.invoke("What is the protocol for managing sepsis in a critical care unit?")

relevant_documents

[Document(metadata={'creationDate': 'D:20120615054440Z', 'trapped': '', 'format': 'PDF 1.7', 'keywords': '', 'subject': '', 'page': 2400, 'creator': 'Atop CHM to PDF Converter', 'moddate': '2026-03-25T04:32:29+00:00', 'total_pages': 4114, 'author': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'modDate': 'D:20260325043229Z', 'file_path': '/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'source': '/content/drive/MyDrive/AIML/medical_diagnosis_manual.pdf'}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special\npopulations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients

In [47]:
model_output = llm(
  "What is the protocol for managing sepsis in a critical care unit?",
    max_tokens=512,
    temperature=0,
    echo=False
)


Llama.generate: prefix-match hit


In [48]:
model_output['choices'][0]['text'].strip()

'Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if clinical

### System and User Prompt Template

In [49]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical guidelines and best practices.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references t

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [50]:
qna_user_message_template = """
### Context
Here are some excerpts from the medical manual:
{context}

### Question
{question}

"""

### Response Function

In [69]:
def generate_rag_response(user_input,k=1,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template, vector_store
    # Create a temporary retriever with the desired k
    current_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": k})
    relevant_document_chunks = current_retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [83]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input_1))

Llama.generate: prefix-match hit


Answer:
The context does not provide specific information on the protocol for managing sepsis in a critical care unit. However, it mentions that prevention of infection is an important aspect of supportive care for ICU patients (p. 3480). Early detection and treatment of sepsis are crucial to improve patient outcomes.

Source:
The context does not provide a specific source for information on the protocol for managing sepsis in a critical care unit.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [84]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input_2))

Llama.generate: prefix-match hit


Answer:
The common symptoms for appendicitis include abdominal pain, usually localized in the right lower quadrant, loss of appetite, nausea, vomiting, constipation or diarrhea, and fever. Appendicitis cannot be cured via medicine alone; it requires surgical removal of the appendix (appendectomy).

Source:
The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 11. Acute Abdomen & Surgical Gastroenterology, pages 165-167.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [85]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input_3))

Llama.generate: prefix-match hit


Answer:
The treatment options for sudden patchy hair loss, also known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA) [Context: Table 86-2. Interpreting Findings in Alopecia]. Hormonal modulators such as oral contraceptives or spironol


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [86]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input_4))

Llama.generate: prefix-match hit


Answer:
The initial treatment for traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas (Chapter 324, The Merck Manual of Diagnosis & Therapy, p. 3395). In the first few days after the injury, maintaining adequate brain perfusion and oxygen


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [87]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input_5))

Llama.generate: prefix-match hit


Answer:
For a person with a fractured leg from a hiking accident, it's crucial to immobilize the affected limb using a splint or cast. This will help prevent further damage and allow the bone to heal properly. The patient should also apply ice to reduce swelling and pain. Seek medical care if an odor emanates from within the cast, a fever develops, or there is suspicion of infection (Merck Manual, 3382). Prolonged immobilization can lead to complications such as stiffness, contractures, and muscle atrophy;


## Fine-tuning

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [88]:
fn_user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(fn_user_input_1, temperature=0.5))

Llama.generate: prefix-match hit


Answer:
The context does not provide specific information on managing sepsis in a critical care unit. However, it mentions that preventing infection is an essential part of supportive care for ICU patients (see p. 3480). Early detection and prompt treatment of sepsis are crucial to improve patient outcomes. Standard sepsis management includes administering appropriate antibiotics based on culture results, fluid resuscitation, and addressing underlying conditions.

Source:
Not applicable in this case as the context does not provide a specific reference for managing sepsis in the ICU.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [89]:
fn_user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(fn_user_input_2,top_p=0.8,top_k=20))

Llama.generate: prefix-match hit


Answer:
The common symptoms for appendicitis include abdominal pain, usually localized in the right lower quadrant, loss of appetite, nausea, vomiting, constipation or diarrhea, and fever. Appendicitis cannot be cured via medicine alone; it requires surgical removal of the appendix (appendectomy).

Source:
The Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 11. Acute Abdomen & Surgical Gastroenterology, pages 165-167.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [90]:
fn_user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(fn_user_input_3,max_tokens=200))

Llama.generate: prefix-match hit


Answer:
The treatment options for sudden patchy hair loss, also known as alopecia areata, include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA) [Context: Table 86-2. Interpreting Findings in Alopecia]. Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogenemia [Context: Table 86-2. Interpreting Findings in Alopecia]. If the underlying disorder is identified, it should be treated accordingly [Context: Hair loss due to other causes].

Source:
The Merck Manual of


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [91]:
fn_user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(fn_user_input_4,top_p=0.85,top_k=20,temperature=0.9))

Llama.generate: prefix-match hit


Answer:
The initial treatment for traumatic brain injury (TBI) involves ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas (Chapter 324, The Merck Manual of Diagnosis & Therapy, p. 3395). In the first few days after the injury, maintaining adequate brain perfusion and oxygen


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [97]:
fn_user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(fn_user_input_5))

Llama.generate: prefix-match hit


Answer:
For a person with a fractured leg from a hiking accident, it's crucial to immobilize the affected limb using a splint or cast. This will help prevent further damage and allow the bone to heal properly. The patient should also apply ice to reduce swelling and pain. Seek medical care if an odor emanates from within the cast, a fever develops, or there is suspicion of infection (Merck Manual, 3382). Prolonged immobilization can lead to complications such as stiffness, contractures, and muscle atrophy;


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [93]:
groundedness_rater_system_message  = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.

"""

In [99]:
relevance_rater_system_message = """
Task: Rate relevance of an AI-generated answer

1. Identify key aspects of the question
2. Check if the answer addresses these using the context
3. Verify all important points are included and no irrelevant info is added
4. Briefly explain alignment, missing, or extra details
5. Assign score (1–5):
1 = Not followed, 5 = Fully followed
"""

In [95]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [102]:
def generate_ground_relevance_response(user_input, k=1, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message, qna_user_message_template

    # Step 1: Retrieve relevant chunks and truncate
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content[:800] for d in relevant_document_chunks]
    context_for_query = "\n\n".join(context_list)
    if not context_list:
        return "Sorry, no relevant context found.", "Sorry, no relevant context found."

    # Step 2: Prepare user message
    user_message = qna_user_message_template.format(context=context_for_query, question=user_input)

    # Step 3: Generate answer
    prompt = f"[INST]{qna_system_message}\nuser: {user_message}[/INST]"
    response = llm(prompt=prompt, max_tokens=max_tokens, temperature=temperature,
                   top_p=top_p, top_k=top_k, stop=['INST'])
    answer = response["choices"][0]["text"][:500]  # truncate to avoid overflow

    # Step 4: Prepare groundedness and relevance prompts
    groundedness_prompt = f"[INST]{groundedness_rater_system_message}\nuser: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}[/INST]"
    relevance_prompt = f"[INST]{relevance_rater_system_message}\nuser: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}[/INST]"

    # Step 5: Get ratings
    response_1 = llm(prompt=groundedness_prompt, max_tokens=max_tokens, temperature=temperature,
                     top_p=top_p, top_k=top_k, stop=['INST'])
    response_2 = llm(prompt=relevance_prompt, max_tokens=max_tokens, temperature=temperature,
                     top_p=top_p, top_k=top_k, stop=['INST'])

    return response_1['choices'][0]['text'], response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [103]:
ground, rel = generate_ground_relevance_response(user_input_1, max_tokens=350)
print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps for evaluation:
1. Identify if the answer is derived only from the information presented in the context.
2. Check if the context provides any specific information related to the protocol for managing sepsis in a critical care unit.
3. Determine if the AI generated answer mentions anything outside of the provided context.

Explanation:
The context does not provide any specific information on the protocol for managing sepsis in a critical care unit. The AI generated answer correctly identifies this and does not mention anything outside of the provided context.

Evaluation:
The metric is not followed at all as there is no information in the context about the protocol for managing sepsis, and the AI generated answer does not provide any such information either.

Rating:
1 - The metric is not followed at all.

 ###Answer Evaluation:

1. Identify key aspects of the question: The question asks about the protocol for managing sepsis in a critical care unit.
2. Check if the answer addres

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [104]:
ground, rel = generate_ground_relevance_response(user_input_2, max_tokens=350)
print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key symptoms mentioned in the context for appendicitis.
2. Determine if the AI generated answer includes those symptoms.
3. Check if the AI generated answer mentions that appendicitis cannot be cured via medicine and requires surgical removal.
4. Verify if the surgical procedures mentioned in the answer (open or laparoscopic appendectomy) are consistent with the context.
5. Ensure that the information in the answer is derived only from the context.

Explanation:
The AI generated answer adheres to the metric as it includes the common symptoms for appendicitis mentioned in the context (abdominal pain, usually localized in the right lower quadrant, anorexia, nausea, and vomiting) and correctly states that appendicitis cannot be cured via medicine alone and requires surgical removal. The answer also mentions the specific surgical procedures for treating appendicitis (open or laparoscopic appendectomy), which are consistent with the context.

R

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [105]:
ground, rel = generate_ground_relevance_response(user_input_3, max_tokens=350)
print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information about effective treatments for sudden patchy hair loss from the context.
2. Determine if the AI generated answer is derived only from the information presented in the context.
3. Check if the possible causes of sudden patchy hair loss are mentioned in the context and if they are included in the answer.

Evaluation:
The AI generated answer mentions Finasteride as an effective treatment for certain types of hair loss, which is consistent with the information presented in the context. However, it does not explicitly state that Finasteride is a treatment for sudden patchy hair loss or localized bald spots. Additionally, the possible causes of sudden patchy hair loss are not mentioned in the answer.

Rating:
Based on the evaluation criteria and metric provided, I would rate the answer as follows:
3 - The metric is followed to a good extent. While the answer does not explicitly state that Finasteride is a treatment for sudden pat

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [106]:
ground, rel = generate_ground_relevance_response(user_input_4, max_tokens=350)
print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information in the context related to treatments for traumatic brain injury (TBI).
2. Determine if the AI generated answer includes only the information from the context.
3. Evaluate the extent to which the answer adheres to the metric.

Explanation:
The context provides information about the initial treatment for TBI, including ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may also be needed to treat intracranial pressure or remove hematomas. In the first few days after the injury, maintaining brain perfusion and oxygenation is important.

The AI generated answer includes all of the information from the context related to initial treatments for TBI. Therefore, the answer adheres to the metric as it is derived only from the information presented in the context.

Rating:
Based on the evaluation criteria, I would rate this answer as a 5 because the metric is followed completely.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground, rel = generate_ground_relevance_response(user_input_5, max_tokens=350)
print(ground, end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


## Actionable Insights and Business Recommendations

1. **Improve Contextual Relevance:** Increase the `chunk_overlap` setting in the retriever to enhance the relevance of retrieved results. Because medical manuals often follow step-by-step sequences, a larger overlap helps preserve continuity and context between chunks.

2. **Sustain Strong Groundedness:** The model currently performs well in groundedness due to carefully designed prompts, so maintaining this structured prompting approach is essential.

3. **Enhance Embedding Performance for Medical Use:** Although the existing embedding model is effective, adopting one trained specifically on medical data can further boost retrieval accuracy and domain relevance.

4. **Keep Knowledge Up to Date:** Regular updates to the knowledge base are important to incorporate the latest medical research, protocols, and clinical guidelines, ensuring ongoing accuracy.

5. **Add Multilingual Capabilities:** Introduce support for multiple languages to better serve healthcare professionals across different regions and linguistic backgrounds.

6. **Integrate User Feedback:** Build a feedback system that allows doctors to review and refine responses, helping the chatbot improve based on real-world usage and clinical needs.

7. **Extend to More Medical Fields:** Scale the RAG system to cover additional medical specialties, increasing its usefulness across a wider range of healthcare applications.


<font size=6 color='blue'>Power Ahead</font>
___